### Data Ingestion

In [1]:
###Document Structure

import langchain
import langchain_core
import langchain_community

from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"exmaple.txt",
        "pages":1,
        "author":"S S",
        "date_created":"2026-01-01"
    }
)
doc

Document(metadata={'source': 'exmaple.txt', 'pages': 1, 'author': 'S S', 'date_created': '2026-01-01'}, page_content='this is the main text content I am using to create RAG')

In [3]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [4]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [5]:
### TextLoader
###from langchain.document_loaders import TextLoader

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

c:\PROJECTS\CIRC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [6]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [7]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False
)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Writer', 'creationdate': '2025-04-01T16:51:42+05:30', 'source': '..\\data\\pdf\\DOP.pdf', 'file_path': '..\\data\\pdf\\DOP.pdf', 'total_pages': 293, 'format': 'PDF 1.5', 'title': '', 'author': '43213', 'subject': '', 'keywords': '', 'moddate': '2025-06-24T12:41:04+00:00', 'trapped': '', 'modDate': 'D:20250624124104Z', 'creationDate': "D:20250401165142+05'30'", 'page': 0}, page_content='CREDIT POLICY SECTION\nRISK MANAGEMENT WING\nHEAD  OFFICE  :  BENGALURU-\n560002\nIG No.         : IC/258/2025\nDate            : 01/04/2025\nIndex          : Advances\nSub Index : General\n          SUB: SCHEME OF DELEGATION OF POWERS FOR CREDIT SANCTIONS\n–CONSOLIDATED GUIDELINES FOR FY 2025-26\nSYNOPSIS\n\uf0b7\nConsolidated  guidelines  on  Delegation  of  Powers  for  FY  2025-26-\nUpdated till 31.03.2025\nA well-documented scheme of delegation of powers for credit sanctions is a\npre-requisite for effective 

In [8]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
###from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter
from pathlib import Path

In [10]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf/")

Found 1 PDF files to process

Processing: DOP.pdf
  ✓ Loaded 293 pages

Total documents loaded: 293


In [11]:
all_pdf_documents

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Writer', 'creationdate': '2025-04-01T16:51:42+05:30', 'moddate': '2025-06-24T12:41:04+00:00', 'author': '43213', 'source': '..\\data\\pdf\\DOP.pdf', 'total_pages': 293, 'page': 0, 'page_label': '1', 'source_file': 'DOP.pdf', 'file_type': 'pdf'}, page_content='CREDIT POLICY SECTION\nRISK MANAGEMENT WING\nHEAD  OFFICE  :  BENGALURU-\n560002\nIG No.         : IC/258/2025\nDate            : 01/04/2025\nIndex          : Advances\nSub Index : General\n          SUB: SCHEME OF DELEGATION OF POWERS FOR CREDIT SANCTIONS\n–CONSOLIDATED GUIDELINES FOR FY 2025-26\nSYNOPSIS\n\uf0b7 Consolidated  guidelines  on  Delegation  of  Powers  for  FY  2025-26-\nUpdated till 31.03.2025\nA well-documented scheme of delegation of powers for credit sanctions is a\npre-requisite for effective credit risk management process. In recognition of\nthe requirement, the Bank has put in place a scheme of delegation of powers\nfo

In [12]:
### Text splitting get into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [13]:
chunks=split_documents(all_pdf_documents)
chunks

Split 293 documents into 826 chunks

Example chunk:
Content: CREDIT POLICY SECTION
RISK MANAGEMENT WING
HEAD  OFFICE  :  BENGALURU-
560002
IG No.         : IC/258/2025
Date            : 01/04/2025
Index          : Advances
Sub Index : General
          SUB: SCH...
Metadata: {'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Writer', 'creationdate': '2025-04-01T16:51:42+05:30', 'moddate': '2025-06-24T12:41:04+00:00', 'author': '43213', 'source': '..\\data\\pdf\\DOP.pdf', 'total_pages': 293, 'page': 0, 'page_label': '1', 'source_file': 'DOP.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Writer', 'creationdate': '2025-04-01T16:51:42+05:30', 'moddate': '2025-06-24T12:41:04+00:00', 'author': '43213', 'source': '..\\data\\pdf\\DOP.pdf', 'total_pages': 293, 'page': 0, 'page_label': '1', 'source_file': 'DOP.pdf', 'file_type': 'pdf'}, page_content='CREDIT POLICY SECTION\nRISK MANAGEMENT WING\nHEAD  OFFICE  :  BENGALURU-\n560002\nIG No.         : IC/258/2025\nDate            : 01/04/2025\nIndex          : Advances\nSub Index : General\n          SUB: SCHEME OF DELEGATION OF POWERS FOR CREDIT SANCTIONS\n–CONSOLIDATED GUIDELINES FOR FY 2025-26\nSYNOPSIS\n\uf0b7 Consolidated  guidelines  on  Delegation  of  Powers  for  FY  2025-26-\nUpdated till 31.03.2025\nA well-documented scheme of delegation of powers for credit sanctions is a\npre-requisite for effective credit risk management process. In recognition of\nthe requirement, the Bank has put in place a scheme of delegation of powers\nfo

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 430.47it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [16]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [17]:
chunks

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Writer', 'creationdate': '2025-04-01T16:51:42+05:30', 'moddate': '2025-06-24T12:41:04+00:00', 'author': '43213', 'source': '..\\data\\pdf\\DOP.pdf', 'total_pages': 293, 'page': 0, 'page_label': '1', 'source_file': 'DOP.pdf', 'file_type': 'pdf'}, page_content='CREDIT POLICY SECTION\nRISK MANAGEMENT WING\nHEAD  OFFICE  :  BENGALURU-\n560002\nIG No.         : IC/258/2025\nDate            : 01/04/2025\nIndex          : Advances\nSub Index : General\n          SUB: SCHEME OF DELEGATION OF POWERS FOR CREDIT SANCTIONS\n–CONSOLIDATED GUIDELINES FOR FY 2025-26\nSYNOPSIS\n\uf0b7 Consolidated  guidelines  on  Delegation  of  Powers  for  FY  2025-26-\nUpdated till 31.03.2025\nA well-documented scheme of delegation of powers for credit sanctions is a\npre-requisite for effective credit risk management process. In recognition of\nthe requirement, the Bank has put in place a scheme of delegation of powers\nfo

In [18]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 826 texts...


Batches: 100%|██████████| 26/26 [00:42<00:00,  1.63s/it]


Generated embeddings with shape: (826, 384)
Adding 826 documents to vector store...
Successfully added 826 documents to vector store
Total documents in collection: 826


In [19]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [20]:
rag_retriever

In [21]:
rag_retriever.retrieve("what are the Guidelines for issuance of Bank guarantee")

Retrieving documents for query: 'what are the Guidelines for issuance of Bank guarantee'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.05it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_9d17a972_265',
  'content': 'empowered to permit such extensions on a case-to-case basis. All other guidelines \non issuance of bank Guarantees shall remain unchanged. \n \nThese guidelines are not applicable for Bank Guarantees with 100% margin where \nrespective sanctioning authority may permit multiple extensions. Where the \nguarantee is issued with 100% margin in c ash or our own term deposits, \nextensions may be permitted for further periods even beyond 2 years at the branch \nlevel itself. \n3.20 Guidelines on Advance Payment Guarantees \nThe Advance Payment/Mobilization Advance Guarantee shall come into effect only \nupon recei pt of the Advance Payment/ Mobilization Advance into the operative \naccount of the customer maintained with the branch issuing such Advance \nPayment/ Mobilization Advance Guarantee. The operative account number of our \ncustomer along with IFSC/ SWIFT code of the branch should be incorporated in \nthe body of the Bank Guarantee.',
  'meta

In [22]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

None


In [23]:
from langchain_groq import ChatGroq
###from langchain.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate
###from langchain.schema import HumanMessage, SystemMessage
from langchain_core.messages import HumanMessage, SystemMessage

In [24]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.3-70b-versatile", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [25]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Please set your GROQ_API_KEY environment variable to use the LLM.


In [26]:
rag_retriever.retrieve("what are the Guidelines for issuance of Bank guarantee")

Retrieving documents for query: 'what are the Guidelines for issuance of Bank guarantee'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.47it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_9d17a972_265',
  'content': 'empowered to permit such extensions on a case-to-case basis. All other guidelines \non issuance of bank Guarantees shall remain unchanged. \n \nThese guidelines are not applicable for Bank Guarantees with 100% margin where \nrespective sanctioning authority may permit multiple extensions. Where the \nguarantee is issued with 100% margin in c ash or our own term deposits, \nextensions may be permitted for further periods even beyond 2 years at the branch \nlevel itself. \n3.20 Guidelines on Advance Payment Guarantees \nThe Advance Payment/Mobilization Advance Guarantee shall come into effect only \nupon recei pt of the Advance Payment/ Mobilization Advance into the operative \naccount of the customer maintained with the branch issuing such Advance \nPayment/ Mobilization Advance Guarantee. The operative account number of our \ncustomer along with IFSC/ SWIFT code of the branch should be incorporated in \nthe body of the Bank Guarantee.',
  'meta

In [28]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [29]:
answer=rag_simple("guidelines for issuance of bank guarantee?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'guidelines for issuance of bank guarantee?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.92it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The guidelines for issuance of bank guarantees include: 

1. All other guidelines on issuance of bank guarantees remain unchanged, except for extensions on a case-to-case basis.
2. For 100% margin guarantees, multiple extensions may be permitted, and extensions beyond 2 years can be allowed at the branch level.
3. Advance Payment/Mobilization Advance Guarantees are effective only upon receipt of the advance payment into the customer's operative account.
4. Branches headed by Scale I, II, III, IV, & V may permit certain types of guarantees within their sanctioning powers if fully secured by 100% margin.
5. Delegation of powers for permitting Bank Guarantees with specific clauses, such as operative clauses and jurisdiction clauses, are specified for different authorities.


In [30]:
answer=rag_simple("what is Consortium advances – approval of assessment of working capital limits?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'what is Consortium advances – approval of assessment of working capital limits?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.48it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Consortium advances – approval of assessment of working capital limits is the approval process for assessing working capital limits in respect of consortium accounts, where the bank is not the leader, and the account falls under the sanctioning powers of CACs at HO. The Head of Circle CACs has been delegated with the power to approve the assessment of working capital limits in such cases.


In [31]:
answer=rag_simple("what is Consortium advances approval of assessment of working capital limits?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'what is Consortium advances approval of assessment of working capital limits?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.36it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Consortium advances approval of assessment of working capital limits is delegated to Head of Circle CACs (CGM-CO-CAC/GM-CO-CAC/DGM-CO-CAC) when the bank is not the leader, and to the appropriate sanctioning authority when the bank is the leader. The approval is provisional and subject to ultimate sanction by the appropriate authority.


In [32]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.47it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [33]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what are the what are the Guidelines for issuance of Bank guarantee?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what are the what are the Guidelines for issuance of Bank guarantee?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.40it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
empowered to permit such extensions on a case-to-case basis. All other guidelines 
on issuance of bank Guarantees shall remain unchanged. 
 
These guidelines are not applicable for Bank Guarantees with 100% margin where 
respective sanctioning authori

ty may permit multiple extensions. Where the 
guarantee is issued with 100% margin in c ash or our own term deposits, 
extensions may be permitted for further periods even beyond 2 years at the branch 
level itself. 
3.20 Guidelines on Advance Payment Guarantees 
The Advance Payment/Mobilization Advance Guarantee shall come into effect only 
upon recei pt of the Advance Payment/ Mobilization Advance into the operative 
account of the customer maintained with the branch issuing such Advance 
Payment/ Mobilization Advance Guarantee. The operative account number of our 
customer along with IFSC/ SWIFT code of the branch should be incorporated in 
the body of the Bank Guarantee.

a) The Branch in charge of branches headed by Scale I, II, III, IV & V may permit the 
following types of guarantees within their sanctioning powe rs if they are fully 
secured by 100% margin in cash / term deposits of the Bank. 
 
i. Guarantees where the protective clause is not specified / incorporated.  
ii. Gu